# S1 Staging ? HU Orders

Load the original order export, apply explicit types, retain the latest version per `line_id`, validate reconciliation, and export Parquet. Source currency, cancellations, NULL customer IDs, and unmatched SKUs are preserved.


# 1. Imports & Configuration


In [ ]:
import polars as pl
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "raw_data").is_dir() and (path / "s1_staging").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the FABINI project root from the current directory.")

RAW_DIR = PROJECT_ROOT / "raw_data"
OUTPUT_DIR = PROJECT_ROOT / "s1_staging" / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_FILE = "raw_orders_hu.csv"
OUTPUT_PATH = OUTPUT_DIR / "orders_hu.parquet"
SHOP = "HU"
SOURCE_CURRENCY = "HUF"
EXPECTED_RAW_ROWS = 3101
EXPECTED_STAGED_ROWS = 2955
EXPECTED_RAW_NULL_CUSTOMERS = 168
EXPECTED_STAGED_NULL_CUSTOMERS = 158
EXPECTED_CANCELLED_ROWS = 93
EXPECTED_COLUMNS = ['order_id', 'line_id', 'created_at', 'sku', 'quantity', 'price_with_vat', 'discount_with_vat', 'currency', 'customer_id', 'cancelled_at', 'exported_at']
EXPECTED_STAGED_COLUMNS = EXPECTED_COLUMNS + ["shop"]
REQUIRED_COLUMNS = ["order_id", "line_id", "created_at", "sku", "quantity", "price_with_vat", "discount_with_vat", "currency", "exported_at"]
UNMATCHED_SKUS = ['FB-PAN-24-LTD', 'FB-WOK-30-BLK', 'FB-KNF-UTIL', 'FB-POT-28']


# 2. Load Raw Data

Read every source field as text first so parsing failures can be detected explicitly. Empty CSV cells become NULL; no source rows are filtered.


In [ ]:
RAW_PATH = RAW_DIR / SOURCE_FILE
raw = pl.read_csv(RAW_PATH, infer_schema=False, null_values=[""])
if raw.columns != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected source columns in {SOURCE_FILE}: {raw.columns}")
if raw.height != EXPECTED_RAW_ROWS:
    raise ValueError(f"Expected {EXPECTED_RAW_ROWS} raw rows, found {raw.height} in {SOURCE_FILE}")
missing_required = {column: raw[column].null_count() for column in REQUIRED_COLUMNS if raw[column].null_count()}
if missing_required:
    raise ValueError(f"Required source values are NULL: {missing_required}")
if raw["customer_id"].null_count() != EXPECTED_RAW_NULL_CUSTOMERS:
    raise ValueError(f"Unexpected raw NULL customer count: {raw['customer_id'].null_count()}")
display(raw.head(5))


# 3. Cleaning & Typing

Parse timestamps according to their source representation. `created_at` and `cancelled_at` are UTC; `exported_at` remains timezone-naive because its source has no timezone suffix.


In [ ]:
typed = raw.with_columns([
    pl.col("quantity").cast(pl.Int64, strict=False),
    pl.col("price_with_vat").cast(pl.Decimal(18, 2), strict=False),
    pl.col("discount_with_vat").cast(pl.Decimal(18, 2), strict=False),
    pl.col("created_at").str.to_datetime(format="%Y-%m-%dT%H:%M:%SZ", time_zone="UTC", strict=False),
    pl.col("cancelled_at").str.to_datetime(format="%Y-%m-%dT%H:%M:%SZ", time_zone="UTC", strict=False),
    pl.col("exported_at").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False),
    pl.lit(SHOP, dtype=pl.String).alias("shop"),
])
for column in ["quantity", "price_with_vat", "discount_with_vat", "created_at", "cancelled_at", "exported_at"]:
    if typed[column].null_count() != raw[column].null_count():
        raise ValueError(f"Invalid values found while parsing {column} in {SOURCE_FILE}")
if typed.filter((pl.col("quantity") <= 0) | (pl.col("price_with_vat") < 0) | (pl.col("discount_with_vat") < 0)).height:
    raise ValueError("Orders contain invalid quantity or negative financial values")
if typed.filter(pl.col("currency") != SOURCE_CURRENCY).height:
    raise ValueError(f"Unexpected source currency; expected {SOURCE_CURRENCY}")
if typed["customer_id"].null_count() != raw["customer_id"].null_count():
    raise ValueError("NULL customer_id values were not preserved")
if typed.schema["line_id"] != pl.String or typed.schema["price_with_vat"] != pl.Decimal(18, 2) or typed.schema["discount_with_vat"] != pl.Decimal(18, 2):
    raise TypeError("Order identifiers or financial columns have unexpected types")
if typed.schema["created_at"] != pl.Datetime("us", "UTC") or typed.schema["cancelled_at"] != pl.Datetime("us", "UTC") or typed.schema["exported_at"] != pl.Datetime("us"):
    raise TypeError("Order timestamps have unexpected types")


# 4. Deduplication

Keep the record with the latest `exported_at` for each `line_id`. A tie at the latest timestamp is treated as an unresolved data issue, not selected arbitrarily.


In [ ]:
latest_export = typed.group_by("line_id").agg(pl.col("exported_at").max().alias("_latest_exported_at"))
latest_rows = typed.join(latest_export, on="line_id").filter(pl.col("exported_at") == pl.col("_latest_exported_at"))
latest_ties = latest_rows.group_by("line_id").len().filter(pl.col("len") > 1)
if latest_ties.height:
    raise ValueError(f"Latest exported_at ties found for {latest_ties.height} line_id values")
staged = (
    typed.sort("exported_at", descending=True)
    .unique(subset=["line_id"], keep="first", maintain_order=True)
)
if staged.height != EXPECTED_STAGED_ROWS:
    raise ValueError(f"Expected {EXPECTED_STAGED_ROWS} staged rows, found {staged.height}")
if staged["line_id"].n_unique() != staged.height:
    raise ValueError("line_id is not unique after deduplication")
if not set(UNMATCHED_SKUS).issubset(set(staged["sku"].unique().to_list())):
    raise ValueError("One or more documented unmatched product SKUs are absent from staged orders")
if set(typed["sku"].unique().to_list()) != set(staged["sku"].unique().to_list()):
    raise ValueError("SKU references were lost during deduplication")


# 5. Data Quality & Reconciliation

The raw missing-customer counts come from exploration. Staged NULL counts reflect the latest row per `line_id`; they are checked separately so deduplication does not imply a source NULL loss.


In [ ]:
if staged["customer_id"].null_count() != EXPECTED_STAGED_NULL_CUSTOMERS:
    raise ValueError(f"Expected {EXPECTED_STAGED_NULL_CUSTOMERS} staged NULL customer IDs, found {staged['customer_id'].null_count()}")
if staged.filter(pl.col("cancelled_at").is_not_null()).height != EXPECTED_CANCELLED_ROWS:
    raise ValueError("Staged cancellation count does not match the exploration findings")
if staged.columns != EXPECTED_STAGED_COLUMNS:
    raise ValueError(f"Unexpected staged columns: {staged.columns}")
reconciliation = pl.DataFrame({
    "shop": [SHOP],
    "raw_rows": [raw.height],
    "duplicate_rows_removed": [raw.height - staged.height],
    "staged_rows": [staged.height],
    "raw_null_customer_ids": [raw["customer_id"].null_count()],
    "staged_null_customer_ids": [staged["customer_id"].null_count()],
    "staged_cancelled_rows": [staged.filter(pl.col("cancelled_at").is_not_null()).height],
})
display(reconciliation)
display(staged.head(5))


# 6. Parquet Export

Write the typed, deduplicated order table. The output is read back and checked before the notebook completes.


In [ ]:
staged.write_parquet(OUTPUT_PATH)
exported = pl.read_parquet(OUTPUT_PATH)
if exported.height != EXPECTED_STAGED_ROWS or exported["line_id"].n_unique() != EXPECTED_STAGED_ROWS:
    raise ValueError(f"Parquet export failed row-count or uniqueness validation: {OUTPUT_PATH}")
if exported.schema != staged.schema:
    raise TypeError(f"Parquet schema differs from staged schema: {OUTPUT_PATH}")
display(pl.DataFrame({"output_file": [OUTPUT_PATH.name], "rows": [exported.height], "schema": [str(exported.schema)]}))
